# 7주차 ② CIFAR-10 CNN — 실습 4~6  〔교수용 정답본〕

> ⚠️ **이 파일은 교수용입니다.** 학생에게는 `12_cnn_cifar10_blank.ipynb` 를 배포하세요.

**목표**: `Conv-BN-ReLU-Pool` 블록을 쌓아 CIFAR-10 분류 CNN 을 직접 작성해 학습시키고,
같은 데이터에서 MLP 와 **파라미터 수·정확도를 비교**한다.

| 구조 | 연도 | 핵심 아이디어 | 무엇을 해결했나 |
|---|---|---|---|
| **LeNet-5** | 1998 | `Conv → Pool` 을 번갈아 → FC | CNN 의 기본형을 확립 |
| **VGG** | 2014 | **3×3 커널만** 깊게 (16~19층) | 큰 커널 하나보다 작은 것 여러 개가 낫다 |
| **ResNet** | 2015 | **잔차 연결(skip connection)** | **깊으면 오히려 학습이 안 되는 문제** ★ |

```
   [VGG 의 발견]  5×5 커널 1개  vs  3×3 커널 2개
      보는 범위는 같다.   파라미터는 25개 vs 18개 (적다)
      게다가 활성함수가 2번 들어간다 → 표현력이 더 좋다

   [ResNet — 지름길을 놓는다]
        x ──────────────────────┐
        │                       │ (그대로 더한다)
        └→ Conv → BN → ReLU → Conv → BN →(+)→ ReLU
                                       ↑
                          출력 = F(x) + x
```

> **핵심 메시지 ★ (출제 지점)**: 잔차 연결이 해결한 문제는 **"깊은 망의 학습 실패"** 입니다.
> `+x` 덕분에 **기울기가 지름길로 앞까지 곧장 흐릅니다.**
> 이 `+x` 한 줄이 **11주차 트랜스포머에도 그대로** 들어갑니다.

> `nn.Sequential` 로는 건너뛰는 연결을 못 씁니다 —
> **`nn.Module` 을 상속해 `forward` 를 직접 쓰는 이유**가 여기서 확실해집니다.

## 실습 4 — CIFAR-10 CNN 직접 작성 ★★

In [ ]:
# 셀 1 — CIFAR-10
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치 :", device)

CLASSES = ["비행기","자동차","새","고양이","사슴","개","개구리","말","배","트럭"]
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),  # CIFAR-10 실측
])

full  = datasets.CIFAR10("data", train=True,  download=True, transform=tf)
test  = datasets.CIFAR10("data", train=False, download=True, transform=tf)
train_set, val_set = random_split(full, [45000, 5000],
                                  generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256)
test_loader  = DataLoader(test,      batch_size=256)

xb, yb = next(iter(train_loader))
print("배치 shape :", xb.shape)        # (128, 3, 32, 32) ← 채널이 3

> **관찰 포인트 ★**: FashionMNIST 는 `(B,1,28,28)` 이었습니다. CIFAR-10 은 **`(B,3,32,32)`** —
> **채널이 3개(컬러)** 이고 크기도 큽니다. 그래서 `Conv2d` 의 `in_channels=3` 으로 시작합니다.

> 6주차에 배운 **훈련/검증/테스트 세 분할**을 처음부터 적용했습니다.
> 증강 효과(3교시)를 **검증 곡선으로 판정**해야 하기 때문입니다.

In [ ]:
# 셀 2 — CNN 정의
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # [블록 1]  32×32 → 16×16
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # ★ 크기 유지
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # [블록 2]  16×16 → 8×8
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # [블록 3]  8×8 → 4×4
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),                              # 6주차의 그것
            nn.Linear(128 * 4 * 4, 256),                  # ★ 128×4×4 = 2048
            nn.ReLU(),
            nn.Linear(256, num_classes),                  # ★ Softmax 를 붙이지 않는다
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)

model = CNN().to(device)
print(model)
print("파라미터 수 :", sum(p.numel() for p in model.parameters()))

> **핵심 메시지 ★★ — FC 입력 크기가 이 실습의 핵심입니다.**
> ```
>   입력 32×32 → 블록1 (pool) → 16×16 → 블록2 (pool) → 8×8 → 블록3 (pool) → 4×4
>   마지막 채널 128,  공간 4×4
>   →  Flatten 하면  128 × 4 × 4 = 2048
> ```
> `padding=1, kernel=3` 은 크기를 유지하므로, **줄이는 것은 오직 `MaxPool2d(2)` 세 번**입니다.
> `32 → 16 → 8 → 4`.

In [ ]:
# 셀 3 — 가짜 입력으로 shape 검산  ★ 학습 전에 반드시
dummy = torch.randn(2, 3, 32, 32).to(device)
print("출력 shape :", model(dummy).shape)      # (2, 10) 이어야 한다

# 블록별 출력도 확인해 보자
x = dummy
for name in ["block1", "block2", "block3"]:
    x = getattr(model, name)(x)
    print(f"  {name} 후 : {tuple(x.shape)}")
print(f"  Flatten 하면 : {x.shape[1]} × {x.shape[2]} × {x.shape[3]} = {x[0].numel()}")

> **관찰 포인트 ★**: 블록별 출력이 `(2,32,16,16) → (2,64,8,8) → (2,128,4,4)` 로 나옵니다.

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `mat1 and mat2 shapes cannot be multiplied (2x2048 and 4096x256)` | FC 입력 숫자가 틀렸다. 오류 메시지의 **앞 숫자(2048)** 가 정답 |
> | `Expected 3 channels, got 1` | `in_channels` 를 1 로 뒀다 (FashionMNIST 습관) |
> | `BatchNorm2d` 오류 | `BatchNorm1d` 를 썼다. 이미지는 **2d** |

> **포인트**: 오류 메시지가 **정답을 알려 줍니다.** `2x2048` 의 2048 이 우리가 넣어야 할 값입니다.

> **관찰 포인트**: 파라미터가 약 **60만 개** 나옵니다.
> 1교시에서 *"MLP 첫 층만 157만 개"* 라고 했죠. **CNN 이 전체로도 더 적습니다.**

## 실습 5 — 학습 실행 + 곡선 확인

In [ ]:
# 셀 4 — 학습 (6주차 루프 그대로)
import time

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

EPOCHS = 15          # ★ 실습실 PC 실측에 맞춰 조정 (CPU 는 5)

def evaluate(m, loader):
    m.eval(); c = t = 0; run = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = m(xb)
            run += loss_fn(out, yb).item() * xb.size(0)
            c += (out.argmax(1) == yb).sum().item(); t += yb.size(0)
    return run / t, c / t

tr_hist, va_hist, va_acc = [], [], []
t0 = time.time()

for epoch in range(EPOCHS):
    model.train(); run = n = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        run += loss.item() * xb.size(0); n += xb.size(0)
    scheduler.step()                                  # ★ epoch 끝에 한 번 (6주차)

    vl, va = evaluate(model, val_loader)
    tr_hist.append(run/n); va_hist.append(vl); va_acc.append(va)
    print(f"epoch {epoch+1:2d}/{EPOCHS} | 훈련 {run/n:.4f} | 검증 {vl:.4f} | 검증정확도 {va*100:5.2f}%")

print(f"\n학습 시간 : {time.time()-t0:.1f}초")
print(f"테스트 정확도 : {evaluate(model, test_loader)[1]*100:.2f}%")

In [ ]:
# 셀 5 — 곡선 + 저장
import os
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(tr_hist, label="훈련"); ax[0].plot(va_hist, label="검증")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("손실")
ax[1].plot([a*100 for a in va_acc], marker="o")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%"); ax[1].set_title("검증 정확도")
plt.tight_layout(); plt.show()

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/cnn_cifar10.pt")
print("저장 완료 : models/cnn_cifar10.pt")

> **관찰 포인트 ★**: 6주차에 배운 대로 **두 곡선을 함께** 봅니다.
> 후반에 검증 손실이 올라가기 시작하면 **과적합**입니다 — 3교시의 증강이 그것을 완화합니다.
> **지금 그 갈라짐을 눈에 담아 두세요.** 3교시 비교의 기준선입니다.

> **체크포인트**: 검증 정확도가 **70% 이상** 나오면 통과.

## 실습 6 — MLP 와 비교

In [ ]:
# 셀 6 — 같은 데이터, 같은 조건으로 MLP
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3*32*32, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
).to(device)

opt2 = torch.optim.AdamW(mlp.parameters(), lr=1e-3)
for epoch in range(EPOCHS):
    mlp.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(mlp(xb), yb)
        opt2.zero_grad(); loss.backward(); opt2.step()

mlp_acc = evaluate(mlp, test_loader)[1]
cnn_acc = evaluate(model, test_loader)[1]

print(f"{'':10s} {'파라미터':>12s} {'테스트 정확도':>14s}")
print(f"{'MLP':10s} {sum(p.numel() for p in mlp.parameters()):>12,d} {mlp_acc*100:>13.2f}%")
print(f"{'CNN':10s} {sum(p.numel() for p in model.parameters()):>12,d} {cnn_acc*100:>13.2f}%")

In [ ]:
# 셀 7 — 어디서 틀리나 (클래스별 정확도)
model.eval()
correct = torch.zeros(10); total = torch.zeros(10)
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(1)
        for c in range(10):
            m = yb == c
            total[c] += m.sum().item()
            correct[c] += (pred[m] == c).sum().item()

for c in (correct/total).argsort():
    print(f"  {CLASSES[c]:6s} {correct[c]/total[c]*100:5.1f}%")

> **관찰 포인트 ★★**: **파라미터는 CNN 이 더 적은데 정확도는 더 높습니다.**
> MLP 는 CIFAR-10 에서 대개 50%대에서 막히고, CNN 은 70% 이상 갑니다.
> **1교시에서 말한 두 가지 이유가 숫자로 확인되는 순간**입니다.

> **핵심 메시지 ★**: *"같은 정확도라면 파라미터가 적은 모델이 유리하다"* — 메모리·속도·과적합 모두에서요.
> 그런데 여기서는 **파라미터가 적으면서 정확도도 높습니다.**
> **구조가 문제에 맞으면 그런 일이 일어납니다.** 9주차 전이학습도 같은 이야기입니다.

> 클래스별로 보면 **고양이·개·새**가 어렵고 **자동차·배·트럭**이 쉽습니다.
> 배경이 단순하고 형태가 뚜렷한 쪽이 쉽다는 뜻입니다.

---

### 이 노트북 체크리스트

- [ ] VGG 가 3×3 만 쓰는 이유를 안다
- [ ] **잔차 연결이 해결한 문제**를 말할 수 있다 ★
- [ ] `Conv-BN-ReLU-Pool` 블록을 직접 썼다
- [ ] **FC 입력 크기를 풀링 횟수로 계산**했다 ★★
- [ ] 학습 전에 가짜 입력으로 shape 을 검산했다
- [ ] CIFAR-10 CNN 을 학습시켜 검증 정확도 70% 이상을 얻었다
- [ ] **MLP 와 파라미터 수·정확도를 비교**했다 ★ (과제 제출물)
- [ ] `models/cnn_cifar10.pt` 를 저장했다